
# **Decripcion del Dataset**
### **Predicción del Riesgo de Reingreso Hospitalario - Dataset**

**Descripción:**

Este conjunto de datos reúne información clínica, demográfica y hospitalaria de pacientes con el objetivo de predecir el riesgo de reingreso hospitalario dentro de los 30 días posteriores al alta médica. Su propósito es identificar patrones asociados a pacientes con mayor probabilidad de volver a ser hospitalizados, utilizando variables relacionadas con su historial médico, resultados de laboratorio, tratamientos recibidos y antecedentes de hospitalización.

**Fuente:** Kaggle: [agregar enlace del dataset]

Cada fila representa a un paciente atendido en un contexto hospitalario e incluye información sobre sus características personales, diagnóstico médico, tipo de ingreso, duración de la hospitalización, resultados de pruebas clínicas, medicamentos utilizados y antecedentes de visitas médicas previas.

**Variables principales:**

* ID del paciente
* Edad
* Género
* Etnia
* Estado civil
* Código postal
* Diagnóstico principal
* Diagnóstico secundario
* Presencia de enfermedad crónica
* Puntaje de comorbilidad
* Diabetes
* Hipertensión
* Enfermedad cardíaca
* Tipo de ingreso hospitalario
* Fuente de admisión
* Tipo de alta médica
* Duración de la hospitalización
* Número de hospitalizaciones previas
* Visitas a urgencias en el último año
* Consultas ambulatorias en el último año
* Nivel de glucosa en sangre
* Resultado de HbA1c
* Presión arterial sistólica y diastólica
* Nivel de hemoglobina
* Nivel de creatinina
* Cantidad de procedimientos de laboratorio
* Resultados de laboratorio anormales
* Número de medicamentos recetados
* Uso de insulina
* Cambio de medicación
* Número de procedimientos médicos
* Polifarmacia
* Reingreso hospitalario dentro de 30 días (`readmitted`)

Estos datos permiten analizar factores asociados al reingreso hospitalario, identificar pacientes de alto riesgo y construir modelos predictivos que apoyen la toma de decisiones clínicas. Además, pueden contribuir a mejorar la planificación del alta médica, optimizar el seguimiento posterior del paciente y reducir reingresos hospitalarios evitables.

La variable objetivo del conjunto de datos es **`readmitted`**, donde **1** indica que el paciente fue readmitido dentro de los 30 días posteriores al alta, y **0** indica que no fue readmitido.

El conjunto de datos contiene información relacionada con pacientes hospitalarios, diagnósticos, tratamientos y resultados clínicos. Para completar la descripción con precisión, se debe verificar la cantidad exacta de filas y columnas mediante Python.


### **Importación de Librerías**

In [ ]:
# Importamos pandas para el manejo de tablas (DataFrames)
import pandas as pd

# Importamos numpy para operaciones matemáticas y manejo de valores nulos
import numpy as np

# Importamos matplotlib y seaborn para las visualizaciones y gráficos
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

# Configurar el estilo de los gráficos
sns.set_theme(style="whitegrid")

### **Carga del Conjunto de Datos**

In [ ]:
# Cargamos el dataset indicando el nombre exacto del archivo
###df = pd.read_csv('')

# Primeros 5 registros del conjunto de datos
df.head()

### **Dimensiones del Dataset**

In [ ]:
# df.shape devuelve una tupla: (filas, columnas)
df.shape

### **Tipos de Datos y Estructura Básica**

In [ ]:
# Muestra un resumen del tipo de datos por columna y la memoria utilizada
df.info()

### **Análisis Estadístico Descriptivo**

### **Variables Numéricas**

In [ ]:
# Estadísticas descriptivas para variables numéricas
df.describe().T

### **Variables Categóricas**

In [ ]:
# Distribución de variables categóricas (frecuencias, valores únicos)
df.describe(include='object').T

### **Detección y Corrección de Incongruencias Categóricas**

In [ ]:
# Le decimos a Pandas que no recorte el contenido de las columnas
pd.set_option('display.max_colwidth', None)

# Definimos explícitamente solo las columnas que queremos analizar
columnas_objetivo = ['Gender', 'Membership_Type', 'Favorite_Exercise', 'Churn']

# Creación de la lista para el diagnóstico
lista_resumen = []

for col in columnas_objetivo:
    # Verificamos que la columna exista en el DataFrame para evitar errores
    if col in df.columns:
        cantidad_unicos = df[col].nunique(dropna=False)
        
        # Extraemos TODOS los valores únicos de la columna y los pasamos a una lista
        todos_los_valores = df[col].unique().tolist()
        
        # Convertimos la lista en una cadena de texto separada por comas
        valores_texto = ", ".join([str(valor) for valor in todos_los_valores])
        
        lista_resumen.append({
            'Variable Categórica': col,
            'Valores Únicos Totales': cantidad_unicos,
            'Todos los Valores Únicos': valores_texto
        })

# 3. Generamos y mostramos el DataFrame
df_auditoria = pd.DataFrame(lista_resumen)
display(df_auditoria)

# (Opcional) Restauramos el límite por defecto de Pandas para no afectar futuras tablas
pd.reset_option('display.max_colwidth')

### **Identificación de Valores Nulos, Duplicados y Outliers**

#### **Detección de Valores Nulos**

In [ ]:
nulos_por_columna = df.isnull().sum()
nulos_filtrados = nulos_por_columna[nulos_por_columna > 0]

# Conteo valores Nulos
if nulos_filtrados.empty:
    print("No existen datos Nulos")
else:
    print(nulos_filtrados)

#### **Identificando Duplicados**

In [ ]:
# Identificación de registros duplicados
duplicados = df.duplicated().sum()

print(f"Registros duplicados encontrados: {duplicados}")

if duplicados > 0:
    display(df[df.duplicated()].head())

In [ ]:

#Dado que la variable Name podría corresponder a un apodo o identificador utilizado por los usuarios, se revisaron los nombres repetidos para detectar posibles registros duplicados. Los valores nulos fueron excluidos de este análisis. Posteriormente, se inspeccionaron todas las filas asociadas a los nombres repetidos para verificar si correspondían efectivamente a la misma persona o si simplemente compartían el mismo nombre.
# Buscar posibles duplicados según el nombre (ignorando valores nulos)

# Obtener los nombres que aparecen más de una vez
nombres_repetidos = df["Name"].dropna().value_counts()
nombres_repetidos = nombres_repetidos[nombres_repetidos > 1]

print("Nombres repetidos encontrados:")
display(nombres_repetidos)

# Mostrar todas las filas asociadas a esos nombres
posibles_duplicados = df[
    df["Name"].isin(nombres_repetidos.index)
].sort_values("Name")

print("\nPosibles registros duplicados según el nombre:")
posibles_duplicados

### **Identificando Outliers**

La revisión de valores atípicos (outliers) es importante porque pueden distorsionar el análisis estadístico y afectar el rendimiento de algunos modelos de machine learning. Algoritmos como la regresión lineal, K-Means o KNN son especialmente sensibles a estos valores extremos, mientras que los modelos basados en árboles suelen ser más robustos. Por ello, es recomendable identificarlos y evaluar si corresponden a errores o a comportamientos reales de los datos.

In [ ]:
cols = df.select_dtypes(include='number').columns
colors = sns.color_palette("husl", len(cols))

fig, ax = plt.subplots((len(cols)+2)//3, 3, figsize=(15, 4*((len(cols)+2)//3)))
ax = ax.flatten()

for i, (col, color) in enumerate(zip(cols, colors)):
    sns.boxplot(y=df[col], ax=ax[i], color=color)
    q1, q3 = df[col].quantile([.25, .75])
    iqr = q3 - q1
    n = ((df[col] < q1 - 1.5*iqr) | (df[col] > q3 + 1.5*iqr)).sum()
    ax[i].set_title(f'{col}\nOutliers: {n}')

for j in range(i+1, len(ax)):
    fig.delaxes(ax[j])

plt.tight_layout()
plt.show()

### **Visualizaciones del Análisis Exploratorio**

#### **Distribución de la variable objetivo**

In [ ]:
plt.figure(figsize=(6, 4))

ax = sns.countplot(data=df, x='Churn', palette='Set2')

for p in ax.patches:
    ax.text(
        p.get_x() + p.get_width()/2,
        p.get_height(),
        int(p.get_height()),
        ha='center'
    )

plt.title('Distribución de Abandono (Churn)')
plt.ylabel('Cantidad de Miembros')
plt.show()

### **Correlación entre variables numéricas**

La correlación es uno de los análisis más importantes antes de aplicar cualquier modelo de datos, ya que permite identificar el nivel de relación entre las variables. Esto ayuda a comprender mejor el conjunto de datos y puede orientar la selección de variables y modelos, considerando que no todos los algoritmos se comportan igual frente a distintos tipos de datos.

- **Regresión Lineal:** funciona mejor cuando existe una relación lineal entre las variables. Por ejemplo, predecir el precio de una casa según su tamaño.

* **Árboles de Decisión:** son útiles cuando las relaciones entre las variables son más complejas y no lineales. Por ejemplo, determinar si un cliente comprará un producto según su edad, ingresos y hábitos de compra.

* **K-Nearest Neighbors (KNN):** suele funcionar bien cuando los datos similares tienden a pertenecer a la misma categoría. Por ejemplo, clasificar flores según sus características físicas.

* **Random Forest:** es adecuado para conjuntos de datos con muchas variables y relaciones complejas. Por ejemplo, predecir si un paciente tiene una enfermedad a partir de varios indicadores médicos.

* **Máquinas de Soporte Vectorial (SVM):** pueden ser efectivas cuando las clases están bien separadas. Por ejemplo, distinguir correos electrónicos entre spam y no spam.

In [ ]:
plt.figure(figsize=(10, 8))
# Seleccionamos solo las columnas numéricas para evitar errores de tipo
columnas_numericas = df.select_dtypes(include=[np.number])
correlacion = columnas_numericas.corr()

# Generamos el mapa de calor
sns.heatmap(correlacion, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Mapa de Calor de Correlaciones Numéricas')
plt.show()

### **Resumen de Hallazgos**